In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, plot_stacked_gain_loss_sortable
from modules.visualisations import plot_profile_by_category

from plotly.io import to_html
from IPython.display import display, HTML


In [ ]:
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

## Params

In [ ]:
# # PARMS
# changeable
org_id = 1
start_time = datetime(2025, 1, 20)
end_time = datetime(2025, 1, 29)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

## Load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")
raw_eeg.head()

In [ ]:
eeg_selected_feat = raw_eeg[["time", "mp_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del raw_eeg
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop("mp_id", axis=1).describe())
eeg_selected_feat.head()

In [ ]:
work = eeg_selected_feat[
    (eeg_selected_feat["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (eeg_selected_feat["time"] < pd.Timestamp(end_time, tz='UTC'))
]
del eeg_selected_feat
work.sum(numeric_only=True)

## eda

In [ ]:
def plot_sorted_mps(data:pd.DataFrame, feature:str, show=False) -> None:

    df = data.groupby(by="mp_id").sum(numeric_only=True)[feature]

    df = df.sort_values(ascending=False)
    x = np.arange(1, len(df) + 1)
    y = df.values
    mp_ids = df.index  # for hover

    # uniform colours
    bar_color = 'lightblue'
    hist_color = 'lightblue'
    box_color = 'lightblue'
    border_color = 'black'

    # subplots: 3 rows, 1 column
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.1,
        subplot_titles=(f"Sorted {df.name}-Values", f"Histogramm [{df.name}]", f"Boxplot [{df.name}]")
    )

    # bar plot on top
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        hovertext=mp_ids,
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
        name="Sorted values",
        marker=dict(color=bar_color, line=dict(color=border_color, width=1))
    ), row=1, col=1)

    # histogram in the middle
    fig.add_trace(go.Histogram(
        x=y,
        nbinsx=100,  # adjust number of bins
        name="Histogram",
        marker=dict(color=hist_color, line=dict(color=border_color, width=1)),
    ), row=2, col=1)

    # horizontal boxplot at the bottom
    fig.add_trace(go.Box(
        x=y,
        orientation='h',
        boxpoints='outliers',  # show outliers
        marker=dict(color=border_color),
        line=dict(color=border_color),
        name="Boxplot"
    ), row=3, col=1)

    # layout adjustments
    fig.update_layout(
        title_text=f"{df.name} summed up on single metering points from {start_time.date()} - {end_time.date()} ({len(data.time.unique())} timestamps), {len(df)} mp ids, ",
        template="plotly_white",
        showlegend=False,
        height=900
    )

    # axis titles
    fig.update_xaxes(title_text="sorted Index", row=1, col=1)
    fig.update_yaxes(title_text=df.name, row=1, col=1)
    fig.update_xaxes(title_text=f"{df.name}", row=2, col=1)
    fig.update_yaxes(title_text="count", row=2, col=1)
    fig.update_xaxes(title_text=f"{df.name}", row=3, col=1)
    fig.update_yaxes(title_text="", row=3, col=1)
    
    if show:
        fig.show()
    return fig

In [ ]:
plot_sorted_mps(data=work[work["energy_direction"]=="C"], feature="comm_cov")

### Waterfilling Opt for finding optimal pfs

In [ ]:
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["mp_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_meas_gen", "wt_surp_gen":"sum_surp_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")
agg_on_time.head()

In [ ]:
time_with_deficit = agg_on_time[agg_on_time["sum_surp_gen"] <= 0]
print(f"{len(time_with_deficit)}/{len(agg_on_time)} ({(len(time_with_deficit)/len(agg_on_time)):.2}%) timestamps has deficit. only for consumers during deficit a pf is optimized")
time_with_deficit.head()

In [ ]:
work.head()

#### Acutal optimization calculation

In [ ]:

work_tf_cons = pd.merge(left=work[work["energy_direction"] == 'C'], right=time_with_deficit, on="time", how="inner")
# INIT
work_tf_cons["A_0"] = 0
work_tf_cons["a_0"] = 0
work_tf_cons["r_0"] = work_tf_cons["wt_meas_cons"]
work_tf_cons["R_0"] = work_tf_cons["sum_meas_gen"]
work_tf_cons["U_0"] = True # will this VZP still receive generation at this t?
work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()["U_0"].rename("U_count_0"), on="time", how="left")

i=0
while (work_tf_cons.groupby(by="time").sum()[f"U_{i}"].rename(f"U_count_{i}").max() > 0) & (work_tf_cons[f"R_{i}"].max() > 0):

    work_tf_cons[f"A_{i}"] = work_tf_cons[f"R_{i}"] / work_tf_cons[f"U_count_{i}"]
    work_tf_cons[f"a_{i+1}"] = work_tf_cons[[f"r_{i}", f"A_{i}"]].min(axis=1)
    work_tf_cons[f"r_{i+1}"] = work_tf_cons[f"r_{i}"] - work_tf_cons[f"a_{i+1}"]

    work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()[f"a_{i+1}"].rename(f"sum_a_{i+1}"), on="time", how="left")
    work_tf_cons[f"R_{i+1}"] = work_tf_cons[f"R_{i}"] - work_tf_cons[f"sum_a_{i+1}"]
    work_tf_cons[f"U_{i+1}"] = work_tf_cons[f"r_{i+1}"] > 0
    work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()[f"U_{i+1}"].rename(f"U_count_{i+1}"), on="time", how="left")
    i += 1


# sum up for the final distribution
a_cols = [col for col in work_tf_cons.columns if col.startswith("a_")]
work_tf_cons["cc_opt"] = work_tf_cons[a_cols].sum(axis=1).clip(lower=0)
work_tf_cons["pf"] = work_tf_cons["cc_opt"] /  work_tf_cons["wt_meas_cons"] * 100

tf_schedule = work_tf_cons[["time", "mp_id", "pf"]].copy()
tf_schedule["pf"] = tf_schedule["pf"].fillna(100).clip(upper=100)


#### Evaluation of the algorithm

In [ ]:
gini_for_single_timestamp = []
gini_opt_cc_full_time_horizon = gini(work_tf_cons.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"])
gini_cc_full_time_horizon = gini(work_tf_cons.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])
for act_timestamp in work_tf_cons["time"].unique():
    filtered_dt = work_tf_cons[work_tf_cons["time"] == act_timestamp]
    check = filtered_dt[["time", "mp_id", "wt_meas_cons", "comm_cov", "cc_opt", "pf"]]
    gini_for_single_timestamp.append((act_timestamp, 
                                      gini(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"]), 
                                      gini(check.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"]),
                                      filtered_dt.groupby(by="time").sum(numeric_only=True)["comm_cov"].iloc[0]))

In [ ]:
df = pd.DataFrame(
    gini_for_single_timestamp,
    columns=["time", "gini_cc", "gini_cc*", "cc_sum"]
)
# --- 1) create a complete 15-minute time index ---
full_range = pd.date_range(
    start=df["time"].min(),
    end=df["time"].max(),
    freq="15min"
)

df = df.set_index("time").reindex(full_range)
df.index.name = "time"

# --- 2) create subplot with secondary y-axis ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

# colours
color_cc = "#1f77b4"     # blue
color_opt = "#ff7f0e"    # orange
color_sum = "#2ca02c"    # green

# gini_cc
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["gini_cc"],
    mode="lines",
    name="gini_cc",
    connectgaps=False,
    line=dict(color=color_cc)
), secondary_y=False)

# gini_cc*
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["gini_cc*"],
    mode="lines",
    name="gini_cc*",
    connectgaps=False,
    line=dict(color=color_opt)
), secondary_y=False)

# horizontal lines
fig.add_trace(go.Scatter(
    x=[df.index.min(), df.index.max()],
    y=[gini_cc_full_time_horizon, gini_cc_full_time_horizon],
    mode="lines",
    name="gini_cc_full_time_horizon",
    line=dict(color=color_cc, dash="dash")
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=[df.index.min(), df.index.max()],
    y=[gini_opt_cc_full_time_horizon, gini_opt_cc_full_time_horizon],
    mode="lines",
    name="gini_opt_cc_full_time_horizon",
    line=dict(color=color_opt, dash="dash")
), secondary_y=False)

# new line for cc_sum (right axis)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["cc_sum"],
    mode="lines",
    name="cc_sum",
    line=dict(color=color_sum)
), secondary_y=True)

# layout
fig.update_layout(
    title="Gini Over Time with Fixed Reference Lines and CC Sum",
    xaxis_title="time",
    yaxis_title="gini",
    yaxis2=dict(title="cc_sum", overlaying="y", side="right"),
    template="plotly_white"
)

fig.show()


In [ ]:
def atkinson(x, epsilon=0.5):
    x = np.array(x, dtype=float)
    mean_x = np.mean(x)

    if mean_x == 0:
        return 0.0  # no inequality measurable

    if epsilon == 1:
        # special case: ε = 1 → logarithmic form
        geo_mean = np.exp(np.mean(np.log(x[x > 0])))
        return 1 - (geo_mean / mean_x)
    else:
        # general Atkinson formula
        term = np.mean(x ** (1 - epsilon))
        return 1 - (term ** (1 / (1 - epsilon)) / mean_x)
    
random_value = np.random.choice(work_tf_cons["time"].unique())
print("Random time value:", random_value)

# 2) Filter dt by this value
filtered_dt = work_tf_cons[work_tf_cons["time"] == random_value]
check = filtered_dt[["time", "mp_id", "wt_meas_cons", "comm_cov", "cc_opt", "pf"]]
print(f"gini on original cc: {gini(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])}")
print(f"gini on optimised cc: {gini(check.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"])}")
for act_epsilon in [0.1, 0.2, 0.5, 0.75, 0.99,1, 1.5, 2]:
    print(f"epsilon={act_epsilon}")
    print(f"\tcc: {atkinson(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"], epsilon=act_epsilon)}")
    print(f"\tcc*: {atkinson(check.groupby(by="mp_id").sum(numeric_only=True)["cc_opt"], epsilon=act_epsilon)}")

In [ ]:
# --- empty list for results ---
metrics_list = []

# --- iterate over every quarter-hour ---
for act_timestamp in work_tf_cons["time"].unique():
    filtered_dt = work_tf_cons[work_tf_cons["time"] == act_timestamp]
    grouped = filtered_dt.groupby("mp_id").sum(numeric_only=True)
    
    comm_cov_values = grouped["comm_cov"]
    cc_opt_values = grouped["cc_opt"]

    metrics_list.append({
        "time": act_timestamp,
        "gini_comm_cov": gini(comm_cov_values),
        "gini_cc*": gini(cc_opt_values),
        "atkinson_cc": atkinson(comm_cov_values, epsilon=0.1),
        "atkinson_cc*": atkinson(cc_opt_values, epsilon=0.1)
    })

# --- create DataFrame ---
df_metrics = pd.DataFrame(metrics_list)

# --- optional: sort by time ---
df_metrics = df_metrics.sort_values("time").reset_index(drop=True)

df_metrics.head()


In [ ]:
df = pd.DataFrame(
    df_metrics,
    columns=["time", "atkinson_cc", "atkinson_cc*", "cc_sum"]
)
# --- 1) create a complete 15-minute time index ---
full_range = pd.date_range(
    start=df["time"].min(),
    end=df["time"].max(),
    freq="15min"
)

df = df.set_index("time").reindex(full_range)
df.index.name = "time"

# --- 2) create subplot with secondary y-axis ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

# colours
color_cc = "#1f77b4"     # blue
color_opt = "#ff7f0e"    # orange
color_sum = "#2ca02c"    # green

# gini_cc
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["atkinson_cc"],
    mode="lines",
    name="atkinson_cc",
    connectgaps=False,
    line=dict(color=color_cc)
), secondary_y=False)

# gini_cc*
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["atkinson_cc*"],
    mode="lines",
    name="atkinson_cc*",
    connectgaps=False,
    line=dict(color=color_opt)
), secondary_y=False)

# horizontal lines
fig.add_trace(go.Scatter(
    x=[df.index.min(), df.index.max()],
    y=[gini_cc_full_time_horizon, gini_cc_full_time_horizon],
    mode="lines",
    name="gini_cc_full_time_horizon",
    line=dict(color=color_cc, dash="dash")
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=[df.index.min(), df.index.max()],
    y=[gini_opt_cc_full_time_horizon, gini_opt_cc_full_time_horizon],
    mode="lines",
    name="gini_opt_cc_full_time_horizon",
    line=dict(color=color_opt, dash="dash")
), secondary_y=False)

# new line for cc_sum (right axis)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["cc_sum"],
    mode="lines",
    name="cc_sum",
    line=dict(color=color_sum)
), secondary_y=True)

# layout
fig.update_layout(
    title="Gini Over Time with Fixed Reference Lines and CC Sum",
    xaxis_title="time",
    yaxis_title="gini",
    yaxis2=dict(title="cc_sum", overlaying="y", side="right"),
    template="plotly_white"
)

fig.show()


In [ ]:
# ORIGINAL values
original_vals = plot_sorted_mps(data=check, feature="comm_cov")

# OPTIMIZED values
# focus on diff to prev chart on: 1. x-axis scale, and 2.&3. y-axis scale 
optimized_vals = plot_sorted_mps(data=check,feature="cc_opt")


# Two finished figures
html1 = to_html(original_vals, include_plotlyjs='cdn', full_html=False)
html2 = to_html(optimized_vals, include_plotlyjs=False, full_html=False)

# Both side by side in one flexbox container
html_combined = f"""
<div style="display:flex; flex-direction:row; gap:20px; width:100%;">
    <div style="flex:1;">{html1}</div>
    <div style="flex:1;">{html2}</div>
</div>
"""

display(HTML(html_combined))


#### Converting pf schedule to hourly

In [ ]:
# bring the quarter-hourly PF schedule to hourly, to obtain a valid final OUTPUT for the first time
tf_schedule_hourly = tf_schedule.copy()
tf_schedule_hourly['hour'] = tf_schedule_hourly['time'].dt.floor('h')

# compute the aggregated pf per mp_id and hour
max_pf = (
    tf_schedule_hourly
    .groupby(['mp_id', 'hour'])['pf']
    .transform('max')
)

# overwrite pf
tf_schedule_hourly['pf'] = max_pf

# drop the helper column again once no longer needed
tf_schedule_hourly.drop(columns='hour', inplace=True)

### Apply pf schedule to energy data

In [ ]:
applied_pfs = apply_pf_schedule_to_mps(work, tf_schedule)

#### Summed up for 15min

In [ ]:
applied_pfs["cc_diff"] = applied_pfs["comm_cov"] - applied_pfs["opt_comm_cov"]

check_full_calc_via_time = applied_pfs.groupby(by="time").sum()[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]
check_full_calc_via_time

#### Summed up for single MP over whole time horizon

In [ ]:
check_full_calcc_via_mpid = applied_pfs.groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov", "cc_diff"]]
check_full_calcc_via_mpid.sort_values(by="cc_diff", ascending=False)

#### Summed up for single MP over single timestamp

In [ ]:
single_time_filtered = applied_pfs[ # TODO not fixed 1m5in but random has_surplus 15min timestamp
    (applied_pfs["time"] >= pd.Timestamp(datetime(2025, 6, 22, 2, 30), tz='UTC')) &
    (applied_pfs["time"] < pd.Timestamp(datetime(2025, 6, 22, 2, 45), tz='UTC'))
]

check_full_calc_via_single_timestamp = single_time_filtered[single_time_filtered["energy_direction"]=="C"].groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov", "cc_diff"]]
check_full_calc_via_single_timestamp.sort_values(by="cc_diff", ascending=False)

### Result evaluation

In [ ]:

sums_on_cons_mps = applied_pfs[applied_pfs["energy_direction"] == "C"]

# ORIGINAL values
original_vals = plot_sorted_mps(sums_on_cons_mps, "comm_cov")

# OPTIMIZED values
# focus on diff to prev chart on: 1. x-axis scale, and 2.&3. y-axis scale 
optimized_vals = plot_sorted_mps(sums_on_cons_mps, "opt_comm_cov")


# Two finished figures
html1 = to_html(original_vals, include_plotlyjs='cdn', full_html=False)
html2 = to_html(optimized_vals, include_plotlyjs=False, full_html=False)

# Both side by side in one flexbox container
html_combined = f"""
<div style="display:flex; flex-direction:row; gap:20px; width:100%;">
    <div style="flex:1;">{html1}</div>
    <div style="flex:1;">{html2}</div>
</div>
"""

# display(HTML(html_combined))


In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min

plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, "comm_cov", "opt_comm_cov")

In [ ]:
# TODO cahrt with 2 comparing histgrams, comparing between cc and cc*

In [ ]:
# OPTIMIZATION CHANGES on while time horizon
plot_stacked_gain_loss_sortable(sums_on_cons_mps, "comm_cov", "opt_comm_cov")

#### Explanation of 15min-Opt vs "Full Horizon"-Opt

In [ ]:
obj_ids_to_filter = [238, 134, 162, 25]

temp = applied_pfs[applied_pfs["energy_direction"] == "C"]
#temp = temp[temp["mp_id"].isin(obj_ids_to_filter)]
plot_profile_by_category(temp, energy_col_name='wt_meas_cons', agg_func_str='median', hue_col="mp_id", logo=logo)